<a href="https://colab.research.google.com/github/RahulTechTutorials/Python-Application/blob/master/A_B_Experiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fg-data-profiling --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.3/400.3 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.7 MB/s eta 0:00:00


In [2]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import matplotlib
import datetime
import os
from google.colab import files
import warnings
from data_profiling import ProfileReport
import duckdb as db
from sklearn.linear_model import LinearRegression

warnings.filterwarnings(action='ignore')
%matplotlib inline
pd.set_option('display.max_columns',200)
pd.set_option('display.max_rows',20) ##pd.set_option('display.max_rows',None)
matplotlib.style.use('ggplot')

In [3]:
files.upload()

Saving canva_ab_test_mock.csv to canva_ab_test_mock.csv


{'canva_ab_test_mock.csv': b'user_id,test_group,country,device_type,pre_period_designs,post_period_designs,session_duration_min,converted\n175,Treatment_A,Australia,Desktop,9,13,17.2,1\n114,Treatment_A,United States,Desktop,8,10,7.8,0\n87,Control,Philippines,Desktop,7,5,23.0,0\n240,Treatment_B,Philippines,Mobile,13,15,9.3,0\n13,Control,Philippines,Desktop,9,6,6.6,0\n253,Treatment_B,United Kingdom,Desktop,19,15,23.8,0\n262,Treatment_B,Brazil,Desktop,10,11,23.2,0\n40,Control,Germany,Desktop,8,17,15.7,0\n201,Treatment_B,Philippines,Mobile,7,9,19.8,0\n62,Control,United States,Mobile,11,11,17.0,1\n123,Treatment_A,United Kingdom,Mobile,7,12,16.4,0\n280,Treatment_B,Brazil,Desktop,11,16,19.0,0\n127,Treatment_A,United Kingdom,Tablet,8,13,14.9,0\n185,Treatment_A,Philippines,Mobile,7,16,5.6,0\n152,Treatment_A,India,Desktop,8,12,9.9,0\n189,Treatment_A,United States,Desktop,6,7,12.6,0\n131,Treatment_A,United States,Desktop,8,10,12.9,0\n219,Treatment_B,India,Desktop,3,13,12.8,0\n71,Control,Brazil,De

In [4]:
os.listdir()

['.config', 'canva_ab_test_mock.csv', 'sample_data']

In [5]:
df = pd.read_csv('canva_ab_test_mock.csv')

In [6]:
df.head()

,user_id,test_group,country,device_type,pre_period_designs,post_period_designs,session_duration_min,converted
0,175,Treatment_A,Australia,Desktop,9,13,17.2,1
1,114,Treatment_A,United States,Desktop,8,10,7.8,0
2,87,Control,Philippines,Desktop,7,5,23.0,0
3,240,Treatment_B,Philippines,Mobile,13,15,9.3,0
4,13,Control,Philippines,Desktop,9,6,6.6,0


In [7]:
df.shape

(300, 8)

In [8]:
profile = ProfileReport(df)
profile

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 8/8 [00:00<00:00, 124.02it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
df.head()

,user_id,test_group,country,device_type,pre_period_designs,post_period_designs,session_duration_min,converted
0,175,Treatment_A,Australia,Desktop,9,13,17.2,1
1,114,Treatment_A,United States,Desktop,8,10,7.8,0
2,87,Control,Philippines,Desktop,7,5,23.0,0
3,240,Treatment_B,Philippines,Mobile,13,15,9.3,0
4,13,Control,Philippines,Desktop,9,6,6.6,0


In [10]:
df.groupby('test_group').size()

,0
test_group,
Control,100
Treatment_A,100
Treatment_B,100


In [11]:
df.groupby('country').size()

,0
country,
Australia,57
Brazil,32
Germany,21
India,60
Philippines,39
United Kingdom,25
United States,66


In [14]:
df.groupby(['test_group','country']).size()

test_group   country       
Control      Australia         21
             Brazil             9
             Germany            9
             India             15
             Philippines       10
             United Kingdom     7
             United States     29
Treatment_A  Australia         14
             Brazil            11
             Germany            8
             India             23
             Philippines       13
             United Kingdom    10
             United States     21
Treatment_B  Australia         22
             Brazil            12
             Germany            4
             India             22
             Philippines       16
             United Kingdom     8
             United States     16
dtype: int64

In [19]:
##df.groupby('device_type')['user_id'].agg(['sum','count'])
db.query('Select device_type, count(user_id) user_cnt, count( distinct user_id) distinct_user_cnt from df group by 1').to_df()

,device_type,user_cnt,distinct_user_cnt
0,Desktop,147,147
1,Mobile,119,119
2,Tablet,34,34


In [20]:
db.query('Select test_group, count(user_id) user_cnt, count( distinct user_id) distinct_user_cnt from df group by 1').to_df()

,test_group,user_cnt,distinct_user_cnt
0,Control,100,100
1,Treatment_A,100,100
2,Treatment_B,100,100


In [29]:
df_conv = df.groupby('test_group').agg(total_converted = ('converted','sum'), total_count = ('converted','count'))

In [39]:
##df_conv['conv_rate'] = df_conv['total_converted'] / df_conv['total_count']
df_conv['conv_rate'] = df_conv.apply(lambda x : x['total_converted']/x['total_count'],axis=1)

In [40]:
df_conv

,total_converted,total_count,conv_rate
test_group,,,
Control,11,100,0.11
Treatment_A,8,100,0.08
Treatment_B,15,100,0.15


In [41]:
df.head()

,user_id,test_group,country,device_type,pre_period_designs,post_period_designs,session_duration_min,converted
0,175,Treatment_A,Australia,Desktop,9,13,17.2,1
1,114,Treatment_A,United States,Desktop,8,10,7.8,0
2,87,Control,Philippines,Desktop,7,5,23.0,0
3,240,Treatment_B,Philippines,Mobile,13,15,9.3,0
4,13,Control,Philippines,Desktop,9,6,6.6,0


In [42]:
df.groupby('test_group').agg(avg_pre_period_designs=('pre_period_designs','mean'),avg_post_period_designs = ('post_period_designs','mean') )

,avg_pre_period_designs,avg_post_period_designs
test_group,,
Control,9.68,10.77
Treatment_A,9.84,12.43
Treatment_B,10.47,15.15


In [46]:
df.groupby('test_group')[['pre_period_designs','post_period_designs','session_duration_min','converted']].mean()

,pre_period_designs,post_period_designs,session_duration_min,converted
test_group,,,,
Control,9.68,10.77,14.853,0.11
Treatment_A,9.84,12.43,14.929,0.08
Treatment_B,10.47,15.15,18.188,0.15


In order to analyse this experiment results, we need to bring in some context as in :
1. Whats the business process looks like.
2. Who were the target audience.

Assumptions :
Process :
1. The target audience was customers who signed up on the free product.
2. The trigger criteria for the experimentation was signup on free sku.
3. The conversion flag tells if the customer converted to paid plan after the free trial period expired.
Experimental:
4. There was no difference in the usage before and after the intervention and the post intervention results are purely the result of intervention.
5. The test group allocation was purely random
6. There was no interaction effect between the Control vs Treatement group
7. There is no Sample Ratio mismatch and the Ratio post experiment is as intended.

Objecttive 1 : Does intervention with treatement A and treatement B has any effect on the conversion rate.

H0 = There is a no effect on the conversion rate with the Treatement A/B
H1 : There is significant difference in the conversion rate

Test Type - Pairwise Independent 2 Sample t test for Treatement A / B vs Control

Objective 2 : Does intervention with treatement A/B lead to increased adoption.i.e. Increased number of designs created (Assumption - the time period observed here is same across both the pre and post experimentation period)

H0 : There is no difference in the usage rate between pre and post exp.
H1 : There is significant difference in adoption between pre and post test.

Test type : Paired test for Treatement A & treatment B


Guardrail: Perform a pairwise t test between the pre-exp-design between Control , Treatment A and Treatement B to confirm there is no significant difference between the customer cohorts between Control Ta & Tb


In [47]:
db.query('Select user_id, count(distinct test_group) group_cnt from df group by user_id having count(distinct test_group) > 1')

┌─────────┬───────────┐
│ user_id │ group_cnt │
│  int64  │   int64   │
├─────────┴───────────┤
│       0 rows        │
└─────────────────────┘

In [72]:
pd.set_option('display.max_rows',10)

In [73]:
df.groupby('user_id').size().to_frame().rename(columns={0:'cnt'}).query("cnt > 1 ")

,cnt
user_id,


### Testing for Randomization - Control vs Treatement A

In [74]:
df.head()

,user_id,test_group,country,device_type,pre_period_designs,post_period_designs,session_duration_min,converted
0,175,Treatment_A,Australia,Desktop,9,13,17.2,1
1,114,Treatment_A,United States,Desktop,8,10,7.8,0
2,87,Control,Philippines,Desktop,7,5,23.0,0
3,240,Treatment_B,Philippines,Mobile,13,15,9.3,0
4,13,Control,Philippines,Desktop,9,6,6.6,0


In [79]:
import scipy.stats as stats
pre_design_c = np.array(df.query("test_group=='Control' ")['pre_period_designs'])
pre_design_ta = np.array(df.query("test_group=='Treatment_A' ")['pre_period_designs'])
pre_design_tb = np.array(df.query("test_group=='Treatment_B' ")['pre_period_designs'])

In [81]:
stats.ttest_ind(a=pre_design_c,b=pre_design_ta,random_state=42, alternative='two-sided')

TtestResult(statistic=np.float64(-0.36549464806055776), pvalue=np.float64(0.7151323548317261), df=np.float64(198.0))

In [80]:
stats.ttest_ind(a=pre_design_c,b=pre_design_tb,random_state=42, alternative='two-sided')

TtestResult(statistic=np.float64(-1.7898462493616514), pvalue=np.float64(0.07500677838409421), df=np.float64(198.0))

In [83]:
np.mean(pre_design_c), np.mean(pre_design_tb),np.mean(pre_design_ta)

(np.float64(9.68), np.float64(10.47), np.float64(9.84))

### Observed - Per Experiment Designs between C, Ta& Tb are are not significantly different

Objective 1 - Conversion rate difference between Control vs Treatement A/B

In [101]:
from statsmodels.stats.proportion import proportions_ztest

In [86]:
df.head()

,user_id,test_group,country,device_type,pre_period_designs,post_period_designs,session_duration_min,converted
0,175,Treatment_A,Australia,Desktop,9,13,17.2,1
1,114,Treatment_A,United States,Desktop,8,10,7.8,0
2,87,Control,Philippines,Desktop,7,5,23.0,0
3,240,Treatment_B,Philippines,Mobile,13,15,9.3,0
4,13,Control,Philippines,Desktop,9,6,6.6,0


In [103]:
c_nobs = df.query("test_group=='Control'")['converted'].count()
c_count = df.query("test_group=='Control'")['converted'].sum()

ta_nobs = df.query("test_group=='Treatment_A'")['converted'].count()
ta_count = df.query("test_group=='Treatment_A'")['converted'].sum()

tb_nobs = df.query("test_group=='Treatment_B'")['converted'].count()
tb_count = df.query("test_group=='Treatment_B'")['converted'].sum()


In [104]:
c_nobs, c_count, ta_nobs, ta_count, tb_nobs, tb_count

(np.int64(100),
 np.int64(11),
 np.int64(100),
 np.int64(8),
 np.int64(100),
 np.int64(15))

In [164]:
##proportions_ztest(count=ta_count, nobs=ta_nobs, value = (c_count/c_nobs) - (ta_count/ta_nobs))
print(c_count, ta_count,c_nobs, ta_nobs)
proportions_ztest(count=[c_count, ta_count], nobs=[c_nobs, ta_nobs])

11 8 100 100


(np.float64(0.7234693963343529), np.float64(0.46939154951843576))

In [169]:
proportions_ztest(count=[8, 20], nobs=[100, 100])

(np.float64(-2.4454174378176674), np.float64(0.014468457105117618))

In [171]:
##proportions_ztest(count=tb_count, nobs=tb_nobs, value = (c_count/c_nobs) - (tb_count/tb_nobs))
print(c_count, tb_count,c_nobs, tb_nobs)
proportions_ztest(count=[c_count, tb_count], nobs=[c_nobs, tb_nobs])

11 15 100 100


(np.float64(-0.8410342670623598), np.float64(0.40032873771624766))

### Observation - The p value for Treatement B is greater than 0.05 and hence not significant. This means that **Intervention with we fail to reject the null hypothesis that Treatement A or B has significant effect on Conversion Rate.**

### Observation 2 - Test the Paired ttest between Treatement A and Treatement B

In [108]:
stats.ttest_rel?

In [109]:
df.columns

Index(['user_id', 'test_group', 'country', 'device_type', 'pre_period_designs',
       'post_period_designs', 'session_duration_min', 'converted'],
      dtype='object')

In [111]:
np.array(df.query("test_group=='Control'")['pre_period_designs'])

array([ 7,  9,  8, 11, 15,  8,  9,  9, 15, 10, 10,  5,  5, 12, 12, 10, 13,
       13,  9, 10, 15, 12, 13,  5,  6,  5, 16,  8, 10,  9,  5, 13, 10,  8,
       11, 12, 10, 12,  8,  8, 16,  9, 10, 11, 10, 12,  7, 10, 11, 11,  4,
       12, 11,  4,  9,  8,  8,  3, 10,  8,  8,  6,  9,  5,  9, 10, 10, 15,
        8, 11,  7, 11, 13, 12,  9,  6,  8, 11, 13, 10, 10, 13,  5, 13,  8,
       10, 20,  9,  7, 13,  9,  7,  9,  7, 16, 11,  6, 10,  7,  7])

In [112]:
c_pre = np.array(df.query("test_group=='Control'")['pre_period_designs'])
c_post = np.array(df.query("test_group=='Control'")['post_period_designs'])

ta_pre = np.array(df.query("test_group=='Treatment_A'")['pre_period_designs'])
ta_post = np.array(df.query("test_group=='Treatment_A'")['post_period_designs'])

tb_pre = np.array(df.query("test_group=='Treatment_B'")['pre_period_designs'])
tb_post = np.array(df.query("test_group=='Treatment_B'")['post_period_designs'])



In [115]:
len(c_pre),len(c_post),len(ta_pre),len(ta_post),len(tb_pre),len(tb_post)

(100, 100, 100, 100, 100, 100)

### Paired ttest for Treatement A

In [116]:
stats.ttest_rel(a=ta_pre,b=ta_post,alternative='two-sided')

TtestResult(statistic=np.float64(-4.712680191728237), pvalue=np.float64(7.998172840947248e-06), df=np.int64(99))

### Paired ttest for Treatement B

In [117]:
stats.ttest_rel(a=tb_pre,b=tb_post,alternative='two-sided')

TtestResult(statistic=np.float64(-10.113933738122432), pvalue=np.float64(6.173289261373756e-17), df=np.int64(99))

In [119]:
np.mean(c_pre),np.mean(c_post),np.mean(ta_pre),np.mean(ta_post),np.mean(tb_pre),np.mean(tb_post)

(np.float64(9.68),
 np.float64(10.77),
 np.float64(9.84),
 np.float64(12.43),
 np.float64(10.47),
 np.float64(15.15))

### Paired ttest for Control

In [120]:
stats.ttest_rel(a=c_pre,b=c_post,alternative='two-sided')

TtestResult(statistic=np.float64(-2.4558714628644136), pvalue=np.float64(0.015796624196399177), df=np.int64(99))

Observation - The significant difference in Control for Pre and post Experimentation invalidates the results for Treatement A and Treatement B. **This means that there is some other confounding element which has an underlying effect between Treatement( Intervention) and Outcome(Increase in Design creation) - That effect is probably a function of time or event**

One more assumption I would like to test - that the test group assingment across the country and device_type was purely random

We will do a Chi Square test for this

In [121]:
df.head()

,user_id,test_group,country,device_type,pre_period_designs,post_period_designs,session_duration_min,converted
0,175,Treatment_A,Australia,Desktop,9,13,17.2,1
1,114,Treatment_A,United States,Desktop,8,10,7.8,0
2,87,Control,Philippines,Desktop,7,5,23.0,0
3,240,Treatment_B,Philippines,Mobile,13,15,9.3,0
4,13,Control,Philippines,Desktop,9,6,6.6,0


In [123]:
df.groupby(['test_group','country']).size()

test_group   country       
Control      Australia         21
             Brazil             9
             Germany            9
             India             15
             Philippines       10
                               ..
Treatment_B  Germany            4
             India             22
             Philippines       16
             United Kingdom     8
             United States     16
Length: 21, dtype: int64

In [150]:
df.groupby(['country']).size()

,0
country,
Australia,57
Brazil,32
Germany,21
India,60
Philippines,39
United Kingdom,25
United States,66


In [133]:
(expected_base / np.sum(expected_base)) * observed_control

array([0.19      , 0.10666667, 0.07      , 0.2       , 0.13      ,
       0.08333333, 0.22      ])

array([21,  9,  9, 15, 10,  7, 29])

In [124]:
stats.chisquare?

In [148]:
expected_base = np.array(df.groupby(['country']).size().sort_index())
observed_control = np.array(df.query("test_group == 'Control'").groupby(['country']).size().sort_index())
expected_control = (expected_base / np.sum(expected_base)) * np.sum(observed_control)

observed_ta = np.array(df.query("test_group == 'Treatment_A'").groupby(['country']).size().sort_index())
expected_ta = (expected_base / np.sum(expected_base)) * np.sum(observed_ta)

observed_tb = np.array(df.query("test_group == 'Treatment_B'").groupby(['country']).size().sort_index())
expected_tb = (expected_base / np.sum(expected_base)) * np.sum(observed_tb)

In [159]:
stats.chisquare(f_obs= observed_control,f_exp= expected_control)

Power_divergenceResult(statistic=np.float64(5.4252853067984645), pvalue=np.float64(0.490532587474392))

In [160]:
stats.chisquare(f_obs= observed_ta,f_exp= expected_ta)

Power_divergenceResult(statistic=np.float64(2.297851161995899), pvalue=np.float64(0.8903699861730798))

In [161]:
stats.chisquare(f_obs= observed_tb,f_exp= expected_tb)

Power_divergenceResult(statistic=np.float64(4.468069824911931), pvalue=np.float64(0.6136023294709383))

Since all the pvalues for Chi Squares are greater than 0.05, the assignment across countries is random and there is no significant difference in the assignment percentage

# Final Conclusion

Basic Assumption Test :

1. The Pre experiment design spread is random - Validated
2. The distribution across coutnries. / device type is also random - Validated
3. There is no interaction effect (SUTVA) - Still assumed


**Conversion Rate** - As the p value was greater than 0.05 for both Treatement A and Treatement B, we fail to reject the Null hypothese that there is difference in conversion rate due to either of the intervention.

Recommendation to Business :
We need to increase power in the experiment to get significant results. that could be done by :
1. Increase the sample size
2. Increase the MDE
3. Reduce the variance in the sample data but controlling Country or Device Type


**Product Usage (Design Created)** : Since the control has significant difference between pre and post designs created, this invalidates the complete test. So we cant comment on the increase of designs created due to intervention.



Next Steps - Since we need to control for time effect, We need to get into difference in difference methodology

In [172]:
df.head()

,user_id,test_group,country,device_type,pre_period_designs,post_period_designs,session_duration_min,converted
0,175,Treatment_A,Australia,Desktop,9,13,17.2,1
1,114,Treatment_A,United States,Desktop,8,10,7.8,0
2,87,Control,Philippines,Desktop,7,5,23.0,0
3,240,Treatment_B,Philippines,Mobile,13,15,9.3,0
4,13,Control,Philippines,Desktop,9,6,6.6,0


In [177]:
arr = np.array(df.query("test_group=='Control'")[['pre_period_designs','post_period_designs']]\
.apply(lambda x : x[1] - x[0], axis=1))

In [200]:
ite_dict = {}
for group in ['Control','Treatment_A','Treatment_B']:
  arr_pre = np.array(df.query("test_group==@group")['pre_period_designs'])
  arr_post = np.array(df.query("test_group==@group")['post_period_designs'])
  ITE = arr_post - arr_pre
  CHANGE = np.sum(ITE)/len(arr_pre)
  key = "ide_control" if group =='Control' else "ide_ta" if group == 'Treatment_A' else "ide_tb"
  print(group, CHANGE)
  ite_dict[key] = ITE


Control 1.09
Treatment_A 2.59
Treatment_B 4.68


In [201]:
##ATE
ate_ta_vs_c = np.mean(ite_dict['ide_ta']) - np.mean(ite_dict['ide_control'])
ate_tb_vs_c = np.mean(ite_dict['ide_tb']) - np.mean(ite_dict['ide_control'])
ate_ta_vs_c, ate_tb_vs_c

(np.float64(1.4999999999999998), np.float64(3.59))

In [189]:
ite_dict

{'ide_control': array([  2,   3,  -9,   0,   8,  -7,  -3,  -1,   5,   0,  -1,  -8,  -9,
         -4,   1,   2,  -1,   1,  -2,  -1,   4,   4,  -3,  -5,  -8,  -5,
          2,  -1,  -6,   2,  -8,   2,   4,  -4,  -3,   4,  -1,   4,  -8,
         -6,   2,  -6,  -3,  -5,  -3,   1,   0,   3,   6,   0,  -8,   5,
          0,  -9,  -2,  -2,   0,  -7,  -1,   1,  -3,  -6,  -5,  -3,   1,
        -14,  -5,   7,  -5,   1,  -3,  -6,   5,  -1,  -2,  -7,  -4,   0,
          8,   2,   0,   0,  -8,   6,  -1,   2,   6,   2,   1,   0,   2,
         -1,   1,   0,   6,   6,  -2,   2,  -6,  -1]),
 'ide_ta': array([ -4,  -2,  -5,  -5,  -9,  -4,  -1,  -2,   2,  -6,   4,   0,  -5,
        -15,   9,   6,  -7,   5, -13,   0,  -4,  -6,  -8,  10,  -2,   7,
         -6,  -4,  -4,  -4,  -3,  -2,  -4,  -8,  -7,   6,   3,   5,  -9,
         -2,   7,  -3,  -2,  -9,  -3,  -4,  -6,   0, -14, -10,  -9, -13,
         -2,   5,  -9,  -1,   0,   1,  -7,  -9,  -1,  -6,   4,   5,  -5,
         -8,   3,   1,   3,   6,  -4,  -7,  

Now we will test these Average Treatement Effect (ATE) for 2 Sample Independent ttest

array([  2,   3,  -9,   0,   8,  -7,  -3,  -1,   5,   0,  -1,  -8,  -9,
        -4,   1,   2,  -1,   1,  -2,  -1,   4,   4,  -3,  -5,  -8,  -5,
         2,  -1,  -6,   2,  -8,   2,   4,  -4,  -3,   4,  -1,   4,  -8,
        -6,   2,  -6,  -3,  -5,  -3,   1,   0,   3,   6,   0,  -8,   5,
         0,  -9,  -2,  -2,   0,  -7,  -1,   1,  -3,  -6,  -5,  -3,   1,
       -14,  -5,   7,  -5,   1,  -3,  -6,   5,  -1,  -2,  -7,  -4,   0,
         8,   2,   0,   0,  -8,   6,  -1,   2,   6,   2,   1,   0,   2,
        -1,   1,   0,   6,   6,  -2,   2,  -6,  -1])

In [194]:
stats.ttest_ind(a=ite_dict['ide_control'],b=ite_dict['ide_ta'])

TtestResult(statistic=np.float64(2.1233846763734348), pvalue=np.float64(0.03496322235938509), df=np.float64(198.0))

In [195]:
stats.ttest_ind(a=ite_dict['ide_control'],b=ite_dict['ide_tb'])

TtestResult(statistic=np.float64(5.59909078023623), pvalue=np.float64(7.127274099976978e-08), df=np.float64(198.0))

In [196]:
stats.ttest_ind(a=ite_dict['ide_ta'],b=ite_dict['ide_tb'])

TtestResult(statistic=np.float64(2.90907928680044), pvalue=np.float64(0.004039317572522032), df=np.float64(198.0))

**Conclusion** - Since the p value is less than 0.05, the results are statsig and we can reject the null hypothese. Hence we can conclude there is significant difference between the Treatement A and Treatment B against the Control for Design Creation.

- Treatement A increases the average designs created in the said timeperiod by 1.5,(p=0.0349)
- Treatement B increases the average designs created in the said timeperiod by 3.6, (p=7.127274099976978e-08)

**If we do Bonferroni correction (α/3 ≈ 0.017) Then the Treatement A becomes insignificant however Treatement B is rock solid.**